In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/SPY_15min_2002-01_to_2006-08.csv')
df2 = pd.read_csv('data/SPY_15min_2020-01_to_2022-01.csv')


df['datetime'] = pd.to_datetime(df['Unnamed: 0'], utc=True)
df['datetime'] = df['datetime'].dt.tz_convert("Europe/Berlin")  # Convert to CET timezone
df.set_index('datetime', inplace=True)

df.index = pd.DatetimeIndex(df.index)  # <--- This ensures time-aware index
df.drop(columns=['Unnamed: 0'], inplace=True)

df = df.asfreq('15min')  # Ensure the index is at 15-minute frequency
df.interpolate(method='time', inplace=True)


# Now these will work
df["day_of_week"] = df.index.dayofweek
df["hour_of_day"] = df.index.hour

day_dummies = pd.get_dummies(df["day_of_week"], prefix="day")[["day_1", "day_2", "day_3", "day_4"]]
hour_dummies = pd.get_dummies(df["hour_of_day"], prefix="hour")[["hour_16", "hour_17", "hour_18", "hour_19", "hour_20", "hour_21", "hour_22", "hour_23", "hour_0"]]

df["log_return"] = df["close"].apply(lambda x: np.log(x)).diff()

df = pd.concat([df, day_dummies, hour_dummies], axis=1)
df

#todo filter for correct dates with dummy variables

,open,high,low,close,volume,day_of_week,hour_of_day,log_return,day_1,day_2,...,day_4,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22,hour_23,hour_0
datetime,,,,,,,,,,,,,,,,,,,,,
2002-01-02 15:30:00+01:00,75.2871,75.3002,74.9339,74.9862,1505500.0,2,15,NaN,False,True,...,False,False,False,False,False,False,False,False,False,False
2002-01-02 15:45:00+01:00,74.9666,75.1105,74.8816,74.8881,1228100.0,2,15,-0.001309,False,True,...,False,False,False,False,False,False,False,False,False,False
2002-01-02 16:00:00+01:00,74.8816,75.1367,74.7900,74.9862,2713000.0,2,16,0.001309,False,True,...,False,True,False,False,False,False,False,False,False,False
2002-01-02 16:15:00+01:00,74.9928,74.9993,74.7181,74.8031,891600.0,2,16,-0.002445,False,True,...,False,True,False,False,False,False,False,False,False,False
2002-01-02 16:30:00+01:00,74.7966,74.8227,74.4434,74.5153,645600.0,2,16,-0.003855,False,True,...,False,True,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2006-09-01 00:30:00+02:00,92.3277,92.3277,92.3277,92.3277,400.0,4,0,0.000382,False,False,...,True,False,False,False,False,False,False,False,False,True
2006-09-01 00:45:00+02:00,92.3136,92.3136,92.2924,92.2924,41400.0,4,0,-0.000382,False,False,...,True,False,False,False,False,False,False,False,False,True
2006-09-01 01:00:00+02:00,92.3277,92.3277,92.3277,92.3277,800.0,4,1,0.000382,False,False,...,True,False,False,False,False,False,False,False,False,False


In [2]:
df_news = pd.read_csv('data/WhatMovesMarkets_eventdatabase.csv')
df_news["eventstart_CET"] = pd.to_datetime(df_news["eventstart_CET"], format="%d.%m.%Y %H:%M:%S").dt.tz_localize("Etc/GMT-1")  # Ensure correct timezone
df_news["eventend_CET"] = pd.to_datetime(df_news["eventend_CET"], format="%d.%m.%Y %H:%M:%S").dt.tz_localize("Etc/GMT-1")  # Ensure correct timezone
df_news.set_index('eventstart_CET', inplace=True)
df_news.drop(columns=["eventstart", "eventend"], inplace=True)
df_news.index = df_news.index.floor("15min")  # Round down to the nearest 15 minutes
df_news["window_start"] = np.where(
        df_news["eventend_CET"].notna(),
        df_news.index - pd.Timedelta(minutes=20),
        df_news.index - pd.Timedelta(minutes=15)
    )
df_news["window_end"] = np.where(
        df_news["eventend_CET"].notna(),
        df_news["eventend_CET"] + pd.Timedelta(minutes=20),
        df_news.index + pd.Timedelta(minutes=30)
    )


,name,type,subtype,subsubtype,description,source,scheduled,eventend_CET,window_start,window_end
eventstart_CET,,,,,,,,,,
2002-03-01 07:00:00+01:00,FI Consumer Confidence,Macro Release,FI,Consumer Confidence,NaN,Bloomberg,1,NaT,2002-03-01 06:45:00+01:00,2002-03-01 07:30:00+01:00
2002-03-01 07:30:00+01:00,CH CPI,Macro Release,CH,CPI,NaN,Bloomberg,1,NaT,2002-03-01 07:15:00+01:00,2002-03-01 08:00:00+01:00
2002-03-01 09:00:00+01:00,IT CPI,Macro Release,IT,CPI,NaN,Bloomberg,1,NaT,2002-03-01 08:45:00+01:00,2002-03-01 09:30:00+01:00
2002-03-01 10:30:00+01:00,UK Monetary Aggregates,Macro Release,UK,Monetary Aggregates,NaN,Bloomberg,1,NaT,2002-03-01 10:15:00+01:00,2002-03-01 11:00:00+01:00
2002-03-01 12:00:00+01:00,EA Retail Sales & EA Retail Trade,Macro Release,EA,Retail Sales & EA Retail Trade,NaN,Bloomberg,1,NaT,2002-03-01 11:45:00+01:00,2002-03-01 12:30:00+01:00
...,...,...,...,...,...,...,...,...,...,...
2020-08-31 17:30:00+01:00,US Auction Result Bill,Auction,US Result,Bill,CUSIP 9127964F3,https://www.treasurydirect.gov/instit/annceres...,1,NaT,2020-08-31 17:15:00+01:00,2020-08-31 18:00:00+01:00
2020-08-31 17:30:00+01:00,US Auction Result Bill,Auction,US Result,Bill,CUSIP 912796TU3,https://www.treasurydirect.gov/instit/annceres...,1,NaT,2020-08-31 17:15:00+01:00,2020-08-31 18:00:00+01:00
2020-09-01 02:00:00+01:00,IE Investec Manufacturing PMI,Macro Release,IE,Investec Manufacturing PMI,NaN,Bloomberg,1,NaT,2020-09-01 01:45:00+01:00,2020-09-01 02:30:00+01:00


In [12]:
print(f"creating {len(df_news["subsubtype"].unique())} event dummie variables")
df[f"D_{newstype}"] = df["datetime"] >= df_news["window_start"] & df["datetime"] < df_news["window_end"] for newstype in df_news["subsubtype"].unique()


creating 128 event dummie variables


['ADP Employment Change',
 'Accounts',
 'Actual FDI & CN Contract FDI Cumulative',
 'Ad Hoc Press Release',
 'Asset Purchase Programmes',
 'BRC Sales Like-For-Like',
 'BRC Shop Price Index',
 'Bank Lending and Monetary Aggregates',
 'Bankruptcies',
 'Beige Book',
 'Bill',
 'Bond',
 'Bond or Note',
 'Budget Balance',
 'Building Permits & US Housing Starts',
 'Business and Consumer Confidence',
 'CPI',
 'CPI & GR CPI EU Harmonized',
 'CPI Hesse',
 'CPI and Earnings Data',
 'CPI and Wages',
 'Cabinet Office Indices',
 'Capacity Utilization & JP Industrial Production',
 'Car Prod.',
 'Car Registrations',
 'Car Sales',
 'Chicago Purchasing Manager',
 'Conference Board Indices',
 'Consumer Confidence',
 'Current Account',
 'Current Account Balance',
 'Discount Rate Minutes',
 'Durable Goods',
 'Earnings Data',
 'Eco Watchters Survey',
 'Economic Bulletin',
 'Empire Manufacturing',
 'Employment',
 'Employment Cost Index',
 'Employment Report',
 'Existing Home Sales',
 'FDI',
 'Factory Orders'